In [ ]:
import numpy as np 
import pandas as pd 
import os 
import sys 
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt 
from pmdarima.preprocessing import BoxCoxEndogTransformer

# %matplotlib notebook
sys.path.insert(1,'../../../../arimaclust/')
from arimaclust.models.centroid import ConsistentARMAClustering
from arimaclust.models.neighbours import NearestNeighbourModel


# Read data

In [ ]:
df_usd = pd.read_csv('data/final_usd_2016_2018.csv',index_col='Unnamed: 0')

In [ ]:
df_usd['AMD'].plot()

In [ ]:
df_usd.drop(columns=['VEF','USD'],inplace=True)

In [ ]:


def prepare_data(df,bc=True):
    X = []
    for col in df.columns:
        x = df[col].dropna().values
        if bc:
            x, _ = BoxCoxEndogTransformer(lmbda2=1e-6).fit_transform(x)
        X.append(x)
    return X

def plot_mds(dist_matrix,labels,names):
    from sklearn import manifold 

    mds = manifold.MDS(
        n_components=2,
        max_iter=3000,
        eps=1e-9,
        random_state=42,
        dissimilarity="precomputed",
        n_jobs=-1,
        normalized_stress="auto",
    )
    pos = mds.fit(dist_matrix).embedding_


    nmds = manifold.MDS(
        n_components=2,
        metric=False,
        max_iter=3000,
        eps=1e-12,
        dissimilarity="precomputed",
        random_state=42,
        n_jobs=-1,
        n_init=1,
        normalized_stress="auto",
    )
    pos = nmds.fit_transform(dist_matrix, init=pos)
    fig, ax = plt.subplots(figsize=(12,8))
    ax.scatter(pos[:,0],pos[:,1],c=labels)
    for i in range(df_usd.columns.shape[0]):
        ax.annotate(names[i], (pos[i][0], pos[i][1]),fontsize=8)
        
def plot_clusters(df,labels,k):
    for i in range(k):
        df[df.columns[labels==i]].plot()
        plt.legend(bbox_to_anchor=(0, 1, 1, 0), ncol=2)
        plt.show()

# Fit without diff

In [ ]:
X = prepare_data(df=df_usd)

In [ ]:
k=20
tau=10
max_orders=(3, 3)

In [ ]:
model = ConsistentARMAClustering(k=k, tau=tau, max_orders=max_orders)
model.fit(X=X)

In [ ]:
plot_mds(dist_matrix=model.D,labels=model.labels_,names=df_usd.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler
df_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_usd),columns=df_usd.columns)

plot_clusters(df=df_scaled,labels=model.labels_,k=k)

# Fit with auto differencing 


In [ ]:
k=20
tau=10
max_orders=(3, 3)
d='auto'

model_diff = ConsistentARMAClustering(k=k,max_orders=max_orders,d=d,tau=tau)
model_diff.fit(X)

In [ ]:
plot_mds(dist_matrix=model_diff.D,labels=model_diff.labels_,names=df_usd.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler
df_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_usd),columns=df_usd.columns)

plot_clusters(df=df_scaled,labels=model_diff.labels_,k=k)

In [ ]:
from arimaclust.autoarima.estimation import fit_arma

In [ ]:
from pmdarima.utils import tsdisplay,autocorr_plot

In [ ]:
autocorr_plot(df_usd['CHF'])

In [ ]:
import pmdarima as pm 
from pmdarima.arima import ndiffs

y_log = np.log1p(df_usd['CHF'].pct_change()).dropna()

y = df_usd['CHF'].dropna()
# y_train,y_test = y[:-100], y[-100:]

kpss_diffs = ndiffs(y_train, alpha=0.05, test='kpss', max_d=6)
adf_diffs = ndiffs(y_train, alpha=0.05, test='adf', max_d=6)
n_diffs = max(adf_diffs, kpss_diffs)
print(n_diffs)

m = pm.auto_arima(y_train,information_criterion='bic',d=None,trace=True,seasonal=False)

In [ ]:
from pmdarima.preprocessing import BoxCoxEndogTransformer

y_train_bc, _ = BoxCoxEndogTransformer(lmbda2=1e-6).fit_transform(y_train)
tsdisplay(y_train_bc, lag_max=100)


In [ ]:
m = pm.auto_arima(y_train_bc,trace=True)

In [ ]:
m.summary()

In [ ]:
import pmdarima as pm 
y = df_usd['CHF'].dropna()
y_train,y_test = y[:-100], y[-100:]
m = pm.auto_arima(y_train,start_p=1,start_q=1,information_criterion='bic',d=0,seasonal=True)

In [ ]:
m.summary()

In [ ]:
y_test_bc = 

In [ ]:
from sklearn.metrics import mean_squared_error
from pmdarima.metrics import smape

model = m  # seeded from the model we've already fit

def forecast_one_step():
    fc, conf_int = model.predict(n_periods=1, return_conf_int=True)
    return (
        fc.tolist()[0],
        np.asarray(conf_int).tolist()[0])

forecasts = []
confidence_intervals = []

for new_ob in y_test:
    fc, conf = forecast_one_step()
    forecasts.append(fc)
    confidence_intervals.append(conf)

    # Updates the existing model with a small number of MLE steps
    model.update(new_ob)

print(f"Mean squared error: {mean_squared_error(y_test, forecasts)}")
print(f"SMAPE: {smape(y_test, forecasts)}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 12))

# --------------------- Actual vs. Predicted --------------------------
# axes[0].plot(y_train, color='blue', label='Training Data')
axes[0].plot(y_test.index, forecasts, color='green', marker='o',
             label='Predicted Price')

axes[0].plot(y_test.index, y_test, color='red', label='Actual Price')
axes[0].set_title('Microsoft Prices Prediction')
axes[0].set_xlabel('Dates')
axes[0].set_ylabel('Prices')

# axes[0].set_xticks(np.arange(0, 7982, 1300).tolist(), df['Date'][0:7982:1300].tolist())
axes[0].legend()


# ------------------ Predicted with confidence intervals ----------------
# axes[1].plot(y_train, color='blue', label='Training Data')
axes[1].plot(y_test.index, forecasts, color='green',
             label='Predicted Price')

axes[1].set_title('Prices Predictions & Confidence Intervals')
axes[1].set_xlabel('Dates')
axes[1].set_ylabel('Prices')

conf_int = np.asarray(confidence_intervals)
axes[1].fill_between(y_test.index,
                     conf_int[:, 0], conf_int[:, 1],
                     alpha=0.9, color='orange',
                     label="Confidence Intervals")

# axes[1].set_xticks(np.arange(0, 7982, 1300).tolist(), df['Date'][0:7982:1300].tolist())
axes[1].legend()

## Fit with deafult pmdarima params

In [ ]:
X = prepare_data(df_usd)

In [ ]:
k=20 
tau=20
max_orders=(3, 3)

model_default = ConsistentARMAClustering(k=k)
model_default.fit(X)

In [ ]:
plot_mds(dist_matrix=model_default.D,labels=model_default.labels_,names=df_usd.columns)

In [ ]:
plot_mds(dist_matrix=model_default.D,labels=model_default.labels_,names=df_usd.columns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
cax = ax.matshow(model_default.D, interpolation='nearest')
fig.colorbar(cax)

In [ ]:
df_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_usd),columns=df_usd.columns)
plot_clusters(df_scaled,model_default.labels_,k=k)

## Neirest neighbours

In [ ]:
from arimaclust.models.neighbours import NearestNeighbourModel
nn_model = NearestNeighbourModel(alpha=0.001,metric='precomputed')
nn_model.fit(model_default.D)

In [ ]:
nn_model.labels_.max()

In [ ]:
model_default.D[list(df_usd.columns).index('RON')][list(df_usd.columns).index('EUR')]

In [ ]:
df_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_usd),columns=df_usd.columns)
plot_clusters(df_scaled,nn_model.labels_,k=nn_model.labels_.max().astype(int))

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage,to_tree, fcluster
from scipy.spatial.distance import squareform
dist_df = pd.DataFrame(data=model.D, columns=df_usd.columns)
dist_matrix_flat = squareform(dist_df)
Z = linkage(dist_matrix_flat, 'ward', metric='precomputed')
# l = fcluster(Z, t=0.5, criterion='distance')

In [ ]:

fig = plt.figure(figsize=(25, 10))
dn = dendrogram(Z,labels=df_usd.columns)
plt.savefig('dend.png',dpi=200)
plt.show()


In [ ]:
l = model.labels_

In [ ]:
cluster = df_usd.columns[l==l[list(df_usd.columns).index('AED')]]

In [ ]:
df_usd[cluster].rolling(3).mean().plot()

In [ ]:
np.save('matrix_16_18_arma.npy',model.D)

In [ ]:
df_names[df_names.Code == 'BTC']

In [ ]:
dist_matrix = np.load('matrix_16_18_arma.npy')


In [ ]:
from arimaclust.models.neighbours import NearestNeighbourModel
nn_model = NearestNeighbourModel(alpha=0.15,metric='precomputed')
nn_model.fit(model_default.D)

In [ ]:
l = np.array(nn_model.labels_)
np.unique(l,return_counts=True)

In [ ]:
cluster = df_usd.columns[l==l[list(df_usd.columns).index('AMD')]]
cluster

In [ ]:
df_names[df_names['Code'].isin(cluster)]

# AMD

In [ ]:
df_amd = pd.read_csv('data/amd/final.csv',index_col='Unnamed: 0').iloc[4900:5100]


In [ ]:
df_amd.drop(columns=['AMD'],inplace=True)

In [ ]:
df_amd = df_amd.dropna(axis=1)

In [ ]:
X_amd = prepare_data(df_amd,bc=False)

In [ ]:
process_args = {'max_p':3,
               'max_q':3,
               'd':0
              }

metric_args = {'tau':10}

In [ ]:


model_amd = ConsistentARMAClustering(k=20,metric_args=metric_args,process_args=process_args)
model_amd.fit(X_amd)

In [ ]:
plot_mds(dist_matrix=model_amd.dist_matrix,labels=model_amd.labels_,names=df_amd.columns)

In [ ]:
def plot_clusters(df,labels,k):
    for i in range(k):
        df[df.columns[labels==i]].plot()
        plt.legend(bbox_to_anchor=(1, 1, 0, 0), ncol=2)
        plt.show()

In [ ]:
df_amd_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_amd),columns=df_amd.columns)
plot_clusters(df_amd_scaled,model_amd.labels_,k=20)

In [ ]:
# alpha = 0.05
nn_model = NearestNeighbourModel(alpha=0.08,metric='precomputed')
nn_model.fit(model_amd.dist_matrix)

In [ ]:
plot_clusters(df_amd_scaled,nn_model.labels_,k=nn_model.labels_.max().astype(int))

In [ ]:
model_amd.dist_matrix[list(df_amd.columns).index('CHF')][list(df_amd.columns).index('EUR')]

In [ ]:
df_amd_scaled[['EUR','USD']].plot()

In [ ]:
pd.Series(nn_model.labels_).value_counts().head(15)

In [ ]:
import pandas as pd 
df_names = pd.read_html('../../../../../table.html')[0]

In [ ]:
smb = 'RUB'
cluster_ind = nn_model.labels_[list(df_amd.columns).index(smb)]
cluster = df_amd.columns[nn_model.labels_ == cluster_ind]
print(cluster)

In [ ]:
df_names[df_names.Code.isin(cluster)]

In [ ]:
process_args = {'max_p':3,
               'max_q':3,
               'd':0
              }

metric_args = {'tau':10}

model_amd = ConsistentARMAClustering(k=7,metric_args=metric_args,process_args=process_args)
model_amd.fit(X_amd)

In [ ]:
plot_clusters(df_amd_scaled,model_amd.labels_,k=7)

In [ ]:
df_usd_part = df_usd.iloc[:300]


In [ ]:
X = prepare_data(df_usd_part,bc=False)

In [ ]:
# alpha = 0.05
nn_model = NearestNeighbourModel(alpha=0.08, metric_args=metric_args, process_args=process_args)
nn_model.fit(X)

In [ ]:
df_usd_p_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_usd_part),columns=df_usd.columns)

plot_clusters(df_usd_p_scaled,nn_model.labels_,k=nn_model.labels_.max().astype(int))